# VisABench across models

Evaluate the same four RefCOCO test samples with Qwen3-VL-8B-Instruct, its SFT adapter, LOOK (10:10), Qwen2.5-VL, LLaVA, and Gemma. Models are loaded sequentially to avoid holding multiple sets of weights on the GPU.

Install the [repository environment](../README.md#installation). Set `LOOK_SFT_ADAPTER` and `LOOK_BEST_ADAPTER` to local checkpoint directories before starting Jupyter, or edit their entries below. The original final checkpoints are SFT step 2,848 and LOOK (10:10) step 5,676. Adapter weights are not bundled; [training instructions](../README.md#training) reproduce them.

For a CPU sample/configuration preview, set `LOOK_NOTEBOOK_PREVIEW=1`. Preview mode does not produce model predictions.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vendor/ms-swift").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
WORK = Path(os.environ.get("WORK", ROOT / "work")).expanduser().resolve()
os.environ["WORK"] = str(WORK)
os.environ.setdefault("HF_HOME", str(WORK / "cache/huggingface"))
os.environ.setdefault("MODELSCOPE_CACHE", str(WORK / "cache/modelscope"))
os.environ.setdefault("MAX_PIXELS", "1003520")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
RUN_INFERENCE = os.environ.get("LOOK_NOTEBOOK_PREVIEW", "0") != "1"
from notebook_utils import load_samples, display_samples, display_table
SAMPLES = load_samples(ROOT)
print(f"{len(SAMPLES)} bundled RefCOCO test samples; inference={RUN_INFERENCE}")

## 1. Inspect the samples

These are four actual RefCOCO testA/testB items with original COCO photographs. The sample manifest retains the same mismatched expressions used by the full VisABench evaluation. A mismatched expression can occasionally describe the current image, so those labels are approximate.

In [ ]:
N_SAMPLES = 4
SELECTED = SAMPLES[:N_SAMPLES]
OUTPUT = WORK / "notebooks/visabench_model_comparison"
display_samples(SELECTED)
display_table(SELECTED, ["id", "expr", "mismatch_expr"])

## 2. Configure the models

All six models are enabled. Set `enabled=False` on a model to run a smaller comparison. Missing SFT or LOOK adapters cause an error before model loading; the runner never substitutes the pretrained model for a missing checkpoint.

Adapter paths are read from `LOOK_SFT_ADAPTER` / `LOOK_BEST_ADAPTER`, or discovered under `$WORK/runs/arm1/**/checkpoint-2848` and `$WORK/runs/arm2/**/checkpoint-5676`. The model IDs below match the evaluation runs. Gemma access may require accepting its model license and authenticating with the model host.

In [ ]:
from notebook_utils import default_models, validate_models
MODELS = default_models(WORK)
# To set paths directly:
# MODELS[1]["adapter"] = "/path/to/sft/checkpoint-2848"
# MODELS[2]["adapter"] = "/path/to/look/checkpoint-5676"
display_table(MODELS, ["label", "model", "adapter", "enabled"])

## 3. Evaluate visual evidence

- **real:** the correct image and expression.
- **blind:** the same input sequence, with visual features replaced by image-placeholder embeddings and zero DeepStack features.
- **mismatch:** the same image and an expression from another image.

The prompt is `Does this image contain: {expr}? Answer with Yes or No.` The original scorer reads the first answer-position logits, summing disjoint whole-token Yes and No variants. It saves `p_yes`, the forced-choice answer, and the model's unconstrained top token. A black image is not used for the blind condition.

In [ ]:
from notebook_utils import run_visabench, summarize_visabench
visa_rows = []
if RUN_INFERENCE:
    validate_models(MODELS)
    visa_rows = run_visabench(SELECTED, MODELS, OUTPUT)
else:
    print("Preview mode: model inference is disabled. Configure adapters and unset LOOK_NOTEBOOK_PREVIEW to run.")

## 4. Compare the models

`yes_rate` is the fraction with `p_yes > 0.5`; `gap` is real-image Yes rate minus no-image Yes rate. `top_token_yes_no` shows how often the model's most likely next token is Yes or No.

As in the paper, no-image Yes rates for Qwen2.5-VL and LLaVA are omitted because their image-placeholder embeddings are untrained. Their raw forced-choice values remain in the saved rows and the separate forced-score columns for inspection. Four samples are useful for examining behavior; use all 10,752 items for the paper's table.

In [ ]:
summary = summarize_visabench(visa_rows)
display_table(summary, ["model", "condition", "n", "yes_rate", "mean_p_yes", "top_token_yes_no", "gap"])

## 5. Read individual predictions

Compare the forced-choice answer with the unconstrained top token. Inspect the raw scores for the omitted no-image entries rather than treating them as reliable Yes/No answers.

In [ ]:
display_table(visa_rows, ["model", "id", "condition", "expr", "expected", "forced_answer", "p_yes", "top_token", "reportable"])
display_table([r for r in summary if r["yes_rate"] is None],
              ["model", "condition", "forced_yes_rate", "forced_mean_p_yes", "top_token_yes_no"])
print("Saved outputs:", OUTPUT)

## Full evaluation

The bundled samples require no dataset download. To reproduce the complete table, prepare RefCOCO and COCO as described in [EVALUATION.md](../EVALUATION.md), then use `python -m bench.blind run --full` for each model/checkpoint followed by `python -m bench.blind score`. The notebook uses the same blinding and logit-scoring functions.